In [ ]:
# Cell 1: Simple Optuna hyperparameter tuning for the LSTM used below
# Minimal example — tune LSTM sizes, learning rate and batch size, return best params.
import optuna
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# use the preprocessed arrays expected by the later cell
X = X_tr_all  # (N_samples, n_in, n_features)
y = y_tr_all.reshape((y_tr_all.shape[0], y_tr_all.shape[1]))  # flatten target to (N, n_out)
#X = X_tr_all_pca  # (N_samples, n_in, n_features) from PCA cell
#y = y_tr_all_pca.reshape((y_tr_all_pca.shape[0], y_tr_all_pca.shape[1]))  # flatten target to (N, n_out)


n_in = X.shape[1]
n_feats = X.shape[2]
n_out = y.shape[1]

def build_model(trial):
    l1 = trial.suggest_int("lstm1_units", 32, 128, step=16)
    l2 = trial.suggest_int("lstm2_units", 16, 64, step=16)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    model = Sequential([
        LSTM(l1, return_sequences=True, input_shape=(n_in, n_feats)),
        LSTM(l2, return_sequences=False),
        Dense(n_out, activation='linear')
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse")
    return model

def objective(trial):
    model = build_model(trial)
    batch = trial.suggest_categorical("batch_size", [32, 64, 128])
    # small number of epochs for quick tuning; increase as needed
    history = model.fit(X, y, epochs=3, batch_size=batch, validation_split=0.1, verbose=0)
    val_loss = history.history["val_loss"][-1]
    tf.keras.backend.clear_session()
    return val_loss

study = optuna.create_study(direction="minimize")
# adjust n_trials as you like (10 is a small, quick run)
study.optimize(objective, n_trials=20)

print("Optuna finished. Best validation loss:", study.best_value)
print("Best hyperparameters:", study.best_params)
# you can access study.best_params to set up the final model / replace constants in the next cell


In [ ]:
# Cell (NEW): Optuna hyperparameter tuning for LSTM corrector
# Simple tuning: tunes LSTM units, learning rate and batch size using validation loss (MSE).
import optuna
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# NOTE: This cell expects X_corr, y_corr, n_in, n_feats, n_out and val_split to be defined
# (they are defined in the cell below). Run that cell first, then run this tuning cell.

def build_and_train_model(units, lr, batch_size, epochs=5):
    tf.keras.backend.clear_session()
    model = Sequential([
        LSTM(units, return_sequences=False, input_shape=(n_in, n_feats)),
        Dense(n_out, activation='linear')
    ])
    opt = Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss='mse')
    history = model.fit(
        X_corr, y_corr,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=val_split,
        verbose=0
    )
    # return final validation loss
    return history.history['val_loss'][-1]

def objective(trial):
    units = trial.suggest_int("units", 8, 128, step=8)
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)
    #lr_2 = trial.suggest_loguniform("lr_2", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    # use small number of epochs for quick tuning; 
    val_loss = build_and_train_model(units=units, lr=lr, batch_size=batch_size, epochs=5)
    return val_loss

# run study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

# best results
best_params = study.best_params
best_value = study.best_value
print("Optuna best value (val_loss):", best_value)
print("Optuna best params:", best_params)

# expose best_params for later cells
optuna_best_params = best_params


In [ ]:
# ------------------------------------------------------------------------------
# Hyperparameter Tuning for PINN using Optuna
# ------------------------------------------------------------------------------
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import math
import random
import subprocess
import sys

# Ensure Optuna is installed
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"], stdout=subprocess.DEVNULL)
    import optuna

# Ensure Shap is installed (from original code)
try:
    import shap
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"], stdout=subprocess.DEVNULL)
    import shap

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# ---------------------------
# Global Config & Reproducibility
# ---------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#PCA:
X_tr_all = X_tr_all_pca  # use PCA-transformed data
y_tr_all = y_tr_all_pca  # use PCA-transformed data
if X_tr_all.ndim == 3:
    n_feats = X_tr_all.shape[2]
else:
    n_feats = int(X_tr_all.shape[1] / N_IN) # recover features if flattened

per_subject_train = per_subject_train_pca  # use PCA-transformed data
per_subject_test = per_subject_test_pca    # use PCA-transformed data


# Fixed Physics Constants
dt_minutes = 5.0
dt_scale = dt_minutes / dt_minutes
cg_min = float(cg_scaler.data_min_[0])
cg_max = float(cg_scaler.data_max_[0])
cg_range = cg_max - cg_min if cg_max > cg_min else 1.0
n_feats = 1 + len(feat_cols)

# ---------------------------
# Helper Functions
# ---------------------------
def inv_cgm_scaled_tf(x_scaled):
    x = tf.cast(x_scaled, tf.float32)
    return x * tf.cast(cg_range, tf.float32) + tf.cast(cg_min, tf.float32)

def inv_cgm_scaled_np(x_scaled_np):
    return x_scaled_np * cg_range + cg_min

def _raw_from_positive_init(init_val):
    v = float(max(init_val, 1e-8))
    return float(np.log(np.expm1(v)))

def _positive_params_from_raw(p1_r, p2_r, p3_r, k_r, Gb_r):
    p1v = tf.nn.softplus(p1_r) + 1e-9
    p2v = tf.nn.softplus(p2_r) + 1e-9
    p3v = tf.nn.softplus(p3_r) + 1e-9
    kv  = tf.nn.softplus(k_r) + 1e-9
    Gbv = tf.nn.softplus(Gb_r) + 1e-6
    return p1v, p2v, p3v, kv, Gbv

@tf.function
def bergman_simulate_tf(G0, bolus_mat_tf, carb_mat_tf, p1v, p2v, p3v, k_iv, Gbv):
    # Standard Bergman simulation
    G = tf.cast(G0, tf.float32)
    I = tf.zeros_like(G, dtype=tf.float32)
    out = tf.TensorArray(tf.float32, size=N_OUT)
    exp_k = tf.exp(-k_iv)
    for t in tf.range(N_OUT):
        bol = tf.cast(bolus_mat_tf[:, t], tf.float32)
        carb = tf.cast(carb_mat_tf[:, t], tf.float32)
        I = I * exp_k + bol
        dG = -p1v * (G - Gbv) - p2v * I + p3v * carb
        G = G + dG * dt_scale
        out = out.write(t, G)
    return tf.transpose(out.stack(), perm=[1,0])

@tf.function
def insulin_effect_from_bolus(bolus_batch, k_iv):
    batch = tf.shape(bolus_batch)[0]
    I = tf.zeros((batch,), tf.float32)
    out = tf.TensorArray(tf.float32, size=N_OUT)
    exp_k = tf.exp(-k_iv)
    for t in tf.range(N_OUT):
        I = I * exp_k + tf.cast(bolus_batch[:, t], tf.float32)
        out = out.write(t, I)
    return tf.transpose(out.stack(), perm=[1,0])

# ---------------------------
# Loss Definitions (MSE & Hybrid)
# ---------------------------
a = 1.0; b = 1.0; eps = 1e-6
T_mgdl = 112.5; C = 1.0; w_left = 1.5; w_right = 1.0
min_v = cg_min; scale_range = cg_range; alpha_l1 = 0.6

mse_loss_fn = tf.keras.losses.MeanSquaredError()
rmse_loss_fn = lambda y_true, y_pred: tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)

def _to_mgdl(tensor_scaled):
    return tf.clip_by_value(tensor_scaled * scale_range + min_v, 1e-3, 1000.0)

def _zone_cost(bg_mgdl):
    bg = tf.cast(bg_mgdl, tf.float32)
    diff2 = tf.square(tf.math.log(bg + eps) - tf.math.log(tf.cast(T_mgdl, tf.float32) + eps))
    left_mask = tf.cast(bg < T_mgdl, dtype=bg.dtype)
    return C * (w_left * diff2 * left_mask + w_right * diff2 * (1.0 - left_mask))

def _slope_cost(delta_bg_mgdl):
    delta = tf.cast(delta_bg_mgdl, tf.float32) / 18.0
    neg_mask = tf.cast(delta < 0.0, dtype=delta.dtype)
    return (2.0 * b) * tf.square(delta) * neg_mask + a * tf.square(delta) * (1.0 - neg_mask)

def lstm_paper_loss_hybrid(y_true, y_pred):
    # Hybrid Data Loss
    y_true_mg = _to_mgdl(tf.cast(y_true, tf.float32))
    y_pred_mg = _to_mgdl(tf.cast(y_pred, tf.float32))
    delta_true = y_true_mg - tf.concat([y_true_mg[:, :1] * 0.0, y_true_mg[:, :-1]], axis=1)

    weights = _zone_cost(y_true_mg) + _slope_cost(delta_true) + 1.0
    weights_norm = weights / (tf.reduce_mean(weights) + 1e-6)

    mse_weighted = tf.reduce_mean(weights_norm * tf.square(y_pred_mg - y_true_mg))
    l1_mg = tf.reduce_mean(tf.abs(y_pred_mg - y_true_mg)) / (scale_range + 1e-6)
    loss = mse_weighted + alpha_l1 * l1_mg
    return tf.where(tf.math.is_finite(loss), loss, tf.constant(1e6, dtype=loss.dtype))

@tf.function
def compound_glucose_tf_loss(y_true_scaled, y_pred_scaled):
    # Differentiable Compound Loss
    y_true = _to_mgdl(y_true_scaled)
    y_pred = _to_mgdl(y_pred_scaled)
    
    # RMSE
    rmse = tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)
    
    # Temporal Gain Surrogate (Slope matching)
    dt_true  = y_true[:, 1:] - y_true[:, :-1]
    dt_pred  = y_pred[:, 1:] - y_pred[:, :-1]
    temporal_penalty = tf.reduce_mean(tf.square(dt_pred - dt_true))
    
    # G-Mean Surrogate
    hypo_thr  = tf.constant(70.2,  tf.float32)
    hyper_thr = tf.constant(180.0, tf.float32)
    p_hypo_true = tf.sigmoid(-(y_true - hypo_thr)/10.0)
    p_hyper_true = tf.sigmoid((y_true - hyper_thr)/10.0)
    p_norm_true = 1.0 - p_hypo_true - p_hyper_true
    p_hypo_pred = tf.sigmoid(-(y_pred - hypo_thr)/10.0)
    p_hyper_pred = tf.sigmoid((y_pred - hyper_thr)/10.0)
    p_norm_pred = 1.0 - p_hypo_pred - p_hyper_pred
    recall_hypo  = tf.reduce_mean(p_hypo_pred  * p_hypo_true)
    recall_norm  = tf.reduce_mean(p_norm_pred  * p_norm_true)
    recall_hyper = tf.reduce_mean(p_hyper_pred * p_hyper_true)
    g_mean = tf.exp((tf.math.log(recall_hypo + 1e-8) + tf.math.log(recall_norm + 1e-8) + tf.math.log(recall_hyper + 1e-8)) / 3.0)
    
    #return (rmse / 100.0) + temporal_penalty + (1.0 - g_mean)
    return (rmse / 100.0) + (1.0 - g_mean)
# ---------------------------
# 1. Pre-Processing: Prepare Data & Fit Initial Bergman Params
# ---------------------------
# We do this ONCE before tuning to save time.
print("Preparing Data and Initial Bergman Fit...")

# Construct full arrays from subject dicts
G_inits = []; carb_fut = []; bolus_fut = []; y_true_mg_windows = []
# Also need these aligned with X_tr_all for PINN training
G0_tr_all = []; bolus_tr_all = []; carb_tr_all = []

# Assuming X_tr_all order matches train_ids iteration roughly, 
# but safest to reconstruct based on known Train IDs
# Note: For strict alignment with X_tr_all, we assume X_tr_all was built iterating train_ids
for sid in train_ids:
    info = per_subject_train.get(sid, None)
    if info is None or info['X'].shape[0] == 0: continue
    subj = per_subject[sid]
    # For Bergman Fit
    y_scaled = info['y'].reshape(info['y'].shape[0], N_OUT)
    y_mg = cg_scaler.inverse_transform(y_scaled.reshape(-1,1)).reshape(-1, N_OUT)
    G_init = subj['CGM_smoothed'].values[N_IN-1 : N_IN-1 + info['X'].shape[0]]
    
    bolus_mat = np.zeros((info['X'].shape[0], N_OUT))
    carb_mat = np.zeros((info['X'].shape[0], N_OUT))
    for w in range(info['X'].shape[0]):
        start = N_IN + w
        bolus_mat[w, :] = subj['bolus'].values[start:start+N_OUT]
        carb_mat[w, :] = subj['carbs'].values[start:start+N_OUT]
    
    G_inits.append(G_init)
    carb_fut.append(carb_mat)
    bolus_fut.append(bolus_mat)
    y_true_mg_windows.append(y_mg)
    
    # Store for PINN training
    G0_tr_all.append(G_init)
    bolus_tr_all.append(bolus_mat)
    carb_tr_all.append(carb_mat)

G_inits = np.concatenate(G_inits, axis=0) if G_inits else np.zeros((0,))
carb_fut = np.vstack(carb_fut) if len(carb_fut)>0 else np.zeros((0, N_OUT))
bolus_fut = np.vstack(bolus_fut) if len(bolus_fut)>0 else np.zeros((0, N_OUT))
y_true_mg_windows = np.vstack(y_true_mg_windows) if len(y_true_mg_windows)>0 else np.zeros((0, N_OUT))

G0_tr_all = np.concatenate(G0_tr_all, axis=0)
bolus_tr_all = np.vstack(bolus_tr_all)
carb_tr_all = np.vstack(carb_tr_all)

# --- Initial Bergman Fit ---
init_p1=0.01; init_p2=0.01; init_p3=0.01; init_k_i=0.1
init_Gb = float(np.median(G_inits) if G_inits.size>0 else 100.0)
p1_raw = tf.Variable(_raw_from_positive_init(init_p1), dtype=tf.float32)
p2_raw = tf.Variable(_raw_from_positive_init(init_p2), dtype=tf.float32)
p3_raw = tf.Variable(_raw_from_positive_init(init_p3), dtype=tf.float32)
k_i_raw = tf.Variable(_raw_from_positive_init(init_k_i), dtype=tf.float32)
Gb_raw = tf.Variable(_raw_from_positive_init(max(init_Gb, 1e-3)), dtype=tf.float32)
opt_b = tf.keras.optimizers.Adam(learning_rate=1e-2)

if G_inits.shape[0] > 0:
    ds_b = tf.data.Dataset.from_tensor_slices((
        G_inits.astype(np.float32), bolus_fut.astype(np.float32),
        carb_fut.astype(np.float32), y_true_mg_windows.astype(np.float32)
    )).shuffle(5000).batch(16)
    
    for epoch in range(1): # Keep pre-fit short
        for G0_b, bol_b, carb_b, y_b in ds_b:
            with tf.GradientTape() as tape:
                p1v, p2v, p3v, k_iv, Gbv = _positive_params_from_raw(p1_raw, p2_raw, p3_raw, k_i_raw, Gb_raw)
                pred = bergman_simulate_tf(G0_b, bol_b, carb_b, p1v, p2v, p3v, k_iv, Gbv)
                loss = tf.reduce_mean(tf.square(pred - y_b))
            grads = tape.gradient(loss, [p1_raw, p2_raw, p3_raw, k_i_raw, Gb_raw])
            opt_b.apply_gradients(zip(grads, [p1_raw, p2_raw, p3_raw, k_i_raw, Gb_raw]))

p1_fit, p2_fit, p3_fit, k_i_fit, Gb_fit = _positive_params_from_raw(p1_raw, p2_raw, p3_raw, k_i_raw, Gb_raw)
fitted_bergman_params = (float(p1_fit), float(p2_fit), float(p3_fit), float(k_i_fit), float(Gb_fit))
print("Initial Fitted Bergman Params:", fitted_bergman_params)

# ---------------------------
# 2. Optuna Objective Function
# ---------------------------

# Split Data for Training vs Validation during Tuning
# We split indices to keep X, y, and phys aux arrays aligned
indices = np.arange(X_tr_all.shape[0])
idx_train, idx_val = train_test_split(indices, test_size=0.2, random_state=42)

def create_pinn_model(trial, bergman_inits):
    """Factory to create PINN with tunable architecture."""
    # Hyperparams for Arch
    lstm_units = trial.suggest_int('lstm_units', 32, 128, step=32)
    dense_units = trial.suggest_int('dense_units', 32, 128, step=32)
    
    class PINN_Optuna(tf.keras.Model):
        def __init__(self, n_out):
            super().__init__()
            self.n_out = n_out
            self.encoder = layers.LSTM(lstm_units)
            self.hd1 = layers.Dense(dense_units, activation='relu')
            self.out = layers.Dense(n_out * 2, activation='linear')
            
            # Physics params (trainable)
            self.p1_raw = tf.Variable(_raw_from_positive_init(bergman_inits[0]), dtype=tf.float32, name='p1')
            self.p2_raw = tf.Variable(_raw_from_positive_init(bergman_inits[1]), dtype=tf.float32, name='p2')
            self.p3_raw = tf.Variable(_raw_from_positive_init(bergman_inits[2]), dtype=tf.float32, name='p3')
            self.k_i_raw = tf.Variable(_raw_from_positive_init(bergman_inits[3]), dtype=tf.float32, name='ki')
            self.Gb_raw = tf.Variable(_raw_from_positive_init(max(bergman_inits[4], 1e-3)), dtype=tf.float32, name='Gb')

        def _bergman_params(self):
            return _positive_params_from_raw(self.p1_raw, self.p2_raw, self.p3_raw, self.k_i_raw, self.Gb_raw)

        def call(self, inputs, training=False):
            x = self.encoder(inputs)
            x = self.hd1(x)
            out = self.out(x)
            out = tf.reshape(out, (-1, self.n_out, 2))
            return out[:, :, 0], out[:, :, 1]
            
        @property
        def bergman_param_values(self):
            return self._bergman_params()

    return PINN_Optuna(N_OUT)

def objective(trial):
    tf.keras.backend.clear_session()
    
    # --- Hyperparameters ---
    lr_phase_1 = trial.suggest_float('lr_phase_1', 1e-5, 1e-3, log=True)
    lr_phase_2 = trial.suggest_float('lr_phase_2', 1e-6, 1e-4, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    lambda_phys = trial.suggest_float('lambda_phys', 0.01, 2.0)
    lambda_i_cons = trial.suggest_float('lambda_i_cons', 0.01, 1.0)
    
    # --- Prepare Datasets (using pre-calculated splits) ---
    # Train
    ds_train = tf.data.Dataset.from_tensor_slices((
        X_tr_all[idx_train].astype(np.float32), 
        y_tr_all[idx_train].reshape(len(idx_train), N_OUT).astype(np.float32),
        G0_tr_all[idx_train].astype(np.float32),
        bolus_tr_all[idx_train].astype(np.float32),
        carb_tr_all[idx_train].astype(np.float32)
    )).shuffle(5000).batch(batch_size).prefetch(2)
    
    # Validation
    # For validation, we just need X and y to calc RMSE, but we pass phys data to keep loop structure consistent if needed
    X_val_t = tf.convert_to_tensor(X_tr_all[idx_val].astype(np.float32))
    y_val_t = tf.convert_to_tensor(y_tr_all[idx_val].reshape(len(idx_val), N_OUT).astype(np.float32))

    # --- Build Model ---
    model = create_pinn_model(trial, fitted_bergman_params)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_phase_1)
    #mse_fn = tf.keras.losses.MeanSquaredError()
    hybrid_loss_fn = lstm_paper_loss_hybrid
    huber_loss_fn = tf.keras.losses.Huber(delta=1.0)
    mae_loss_fn = tf.keras.losses.MeanAbsoluteError()
    # --- Training Loop Helper ---
    @tf.function
    def train_step(Xb, yb, G0b, bolb, carb_b, phys_w, data_loss_func):
        with tf.GradientTape() as tape:
            G_pred, I_pred = model(Xb, training=True)
            loss_data = data_loss_func(yb, G_pred)
            
            p1, p2, p3, ki, Gb = model.bergman_param_values
            G_sim = bergman_simulate_tf(G0b, bolb, carb_b, p1, p2, p3, ki, Gb)
            loss_phys = tf.reduce_mean(tf.square(inv_cgm_scaled_tf(G_pred) - G_sim)) / (scale_range**2 + 1e-9)
            loss_icons = tf.reduce_mean(tf.square(I_pred - insulin_effect_from_bolus(bolb, ki)))
            
            total = loss_data + phys_w * loss_phys + lambda_i_cons * loss_icons
        
        grads = tape.gradient(total, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return total

    # --- Phase 1: MSE Training ---
    # Shorten epochs for tuning speed, or use full if resources allow
    epochs_p1 = 5  # Reduced for demonstration/tuning speed. Original was 1? Increase for better tuning.
    for epoch in range(epochs_p1):
        # Warmup phys weight
        pw = float(lambda_phys * min(1.0, (epoch + 1) / 2))
        for Xb, yb, G0b, bolb, carbb in ds_train:
            train_step(Xb, yb, G0b, bolb, carbb, pw, mae_loss_fn)

    # --- Phase 2: Hybrid Training ---
    optimizer.learning_rate.assign(lr_phase_2)
    epochs_p2 = 5 
    for epoch in range(epochs_p2):
        for Xb, yb, G0b, bolb, carbb in ds_train:
            train_step(Xb, yb, G0b, bolb, carbb, lambda_phys, compound_glucose_tf_loss)

    # --- Evaluation ---
    # Calculate Validation RMSE (standard metric)
    G_val_pred, _ = model(X_val_t, training=False)
    # Convert to mg/dL for interpretable error
    val_pred_mg = inv_cgm_scaled_tf(G_val_pred)
    val_true_mg = inv_cgm_scaled_tf(y_val_t)
    val_rmse = tf.sqrt(tf.reduce_mean(tf.square(val_pred_mg - val_true_mg)))
    
    return float(val_rmse.numpy())

# ---------------------------
# 3. Run Optimization
# ---------------------------
print("Starting Optuna Study...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20) # Adjust n_trials as needed

print("\nBest params:")
print(study.best_params)

print("\nRetraining best model on FULL training set...")
best = study.best_params

